In [20]:
import pandas as pd
from sklearn.datasets import fetch_california_housing


In [21]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

In [22]:
df. head()


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [23]:
# seperate features and target
x= df.drop('MedHouseVal',axis= 1)
y= df['MedHouseVal']
print("feature shape: ",x.shape)
print("feature shape: ",y.shape)

feature shape:  (20640, 8)
feature shape:  (20640,)


2. Checking missing values


In [24]:
print(x.isnull().sum())
print("total missing values:", x.isnull().sum().sum())

MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64
total missing values: 0


3. Split training and testing data

In [25]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state= 36
)

4. Handle missing values

In [26]:
from sklearn.impute import SimpleImputer
imputer= SimpleImputer(strategy='median')
x_train_imputed =imputer.fit_transform(x_train)
x_test_imputed = imputer.transform(x_test)
print("feature shape: ",x_train_imputed.shape)
print("feature shape: ",x_test_imputed.shape)


feature shape:  (16512, 8)
feature shape:  (4128, 8)


5. Apply feature scaling

In [27]:
from sklearn.preprocessing import StandardScaler
scaler =StandardScaler()
X_train_scaled= scaler.fit_transform(x_train_imputed)
X_test_scaled = scaler.transform(x_test_imputed)
print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

Scaled training shape: (16512, 8)
Scaled testing shape: (4128, 8)


6. Import LinearRegression model

In [28]:
from sklearn.linear_model import LinearRegression
LR = LinearRegression()

In [29]:
LR.fit(X_train_scaled,y_train)
y_pred= LR.predict(X_test_scaled)


7. Evaluate model

In [30]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
mae =mean_absolute_error(y_test,y_pred)
rmse=mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test,y_pred)

print('mae',mae)
print('rmse',rmse)
print('r2_score',r2)

mae 0.5406753956069983
rmse 0.7293454945633577
r2_score 0.5865144222438183


8. Random forest model

In [31]:
from sklearn.ensemble import RandomForestRegressor
rfr=RandomForestRegressor(
    n_estimators= 100,
    n_jobs=1,
    random_state= 42
)

rfr.fit(x_train,y_train)
y_pred=rfr.predict(x_test)

In [32]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
rf_mae =mean_absolute_error(y_test,y_pred)
rf_rmse=mean_squared_error(y_test, y_pred) ** 0.5
rf_r2 = r2_score(y_test,y_pred)

print('mae',mae)
print('rmse',rmse)
print('r2_score',r2)

mae 0.5406753956069983
rmse 0.7293454945633577
r2_score 0.5865144222438183


In [33]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae, rf_mae],
    "RMSE": [rmse, rf_rmse],
    "R2": [r2, rf_r2]
})

results

,Model,MAE,RMSE,R2
0,Linear Regression,0.540675,0.729345,0.586514
1,Random Forest,0.330367,0.502629,0.803624


9. Save the model

In [35]:
import joblib
from pathlib import Path

Path('../models').mkdir(exist_ok=True)

joblib.dump(
    rfr,
    "../models/house_price_model.joblib"
)
print('model saved succuesfully')

model saved succuesfully


###  Testing

1. Verify the saved model

In [37]:
import joblib

model = joblib.load("../models/house_price_model.joblib")

print("Model loaded successfully!")
print(type(model))

Model loaded successfully!
<class 'sklearn.ensemble._forest.RandomForestRegressor'>


2. Test prediction

In [38]:
sample = x_test.iloc[:3]

predictions = model.predict(sample)

print("Predictions:", predictions)

Predictions: [1.79402   0.63639   4.3738655]


3. Compare actual and predicted values

In [39]:
comparison = pd.DataFrame({
    "Actual": y_test.iloc[:3].values,
    "Predicted": predictions
})

print(comparison)


    Actual  Predicted
0  1.82600   1.794020
1  0.60500   0.636390
2  5.00001   4.373865


In [40]:
import joblib

model = joblib.load(
    "../models/house_price_pipeline.joblib"
)

print(type(model))

<class 'sklearn.pipeline.Pipeline'>
